# Test 3: Ablation Study — Does the DFG Actually Help?

**Goal**: Isolate the contribution of the **Data Flow Graph (DFG)** by comparing:

| Model | Backbone | DFG Attention? |
|---|---|---|
| A — `GraphCodeBERT + DFG` | GraphCodeBERT | ✅ Full DFG-aware sparse mask |
| B — `GraphCodeBERT (no DFG)` | GraphCodeBERT | ❌ Standard causal attention, text-only |

Both models share the **same backbone weights and architecture** — only the
attention mask and position representation differ between runs.
Any performance gap is directly attributable to the DFG.

**Kaggle inputs**:
- `/kaggle/input/dfgdataset2/dataset_graphcodebert.jsonl`
- `/kaggle/input/model2/model.bin` — pre-trained GraphCodeBERT+DFG weights

**Output**: `/kaggle/working/test3_ablation_results.txt`

In [1]:
!pip install torch transformers tree_sitter==0.21.3 scikit-learn matplotlib -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.2/502.2 kB 13.1 MB/s eta 0:00:00


In [2]:
import os, json, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import CrossEntropyLoss
from torch.utils.data import DataLoader, Dataset, random_split
from torch.optim import AdamW
from torch.cuda.amp import autocast, GradScaler
from transformers import (
    get_linear_schedule_with_warmup,
    RobertaConfig, RobertaModel,
    AutoTokenizer)

from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, average_precision_score
)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm import tqdm

print("Imports OK")
print(f"CUDA: {torch.cuda.is_available()}")

Imports OK
CUDA: True


In [3]:
# ─── CONFIGURATION ────────────────────────────────────────────────────────────
class Args:
    train_file          = "/kaggle/input/datasets/hasanmahmudabdullah/dfgdataset2/dataset_graphcodebert.jsonl"
    gcb_weights         = "/kaggle/input/datasets/hasanmahmudabdullah/model2/model.bin"  # trained GraphCodeBERT+DFG
    output_dir_nodeg    = "saved_models_ablation_nodg"       # trained no-DFG model

    model_name_or_path  = "microsoft/graphcodebert-base"
    tokenizer_name      = "microsoft/graphcodebert-base"

    code_length         = 384
    data_flow_length    = 128
    train_batch_size    = 16
    eval_batch_size     = 32
    learning_rate       = 2e-5
    max_grad_norm       = 1.0
    num_train_epochs    = 3
    seed                = 42
    val_ratio           = 0.10    # 90/10 split

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    n_gpu  = torch.cuda.device_count()

args = Args()

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if args.n_gpu > 0:
        torch.cuda.manual_seed_all(seed)

set_seed(args.seed)
print(f"Device: {args.device}")

Device: cuda


In [4]:
# ─── SHARED CLASSIFICATION HEAD (same for both conditions) ───────────────────
class ClassificationModel(nn.Module):
    """Generic head wrapping a RobertaModel encoder."""
    def __init__(self, encoder, config):
        super().__init__()
        self.encoder    = encoder
        self.config     = config
        self.dropout    = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, 2)

    def _classify(self, cls_repr, labels):
        logits = self.classifier(self.dropout(cls_repr))
        prob   = F.softmax(logits, dim=-1)
        if labels is not None:
            return CrossEntropyLoss()(logits, labels), prob
        return prob


# ─── Condition A: full DFG-aware attention ────────────────────────────────────
class ModelWithDFG(ClassificationModel):
    def forward(self, input_ids=None, p_ids=None, attn_mask=None, labels=None):
        ext = (1.0 - attn_mask) * -10000.0
        ext = ext.unsqueeze(1)
        emb = self.encoder.embeddings(input_ids=input_ids, position_ids=p_ids)
        out = self.encoder.encoder(
            emb, attention_mask=ext,
            head_mask=[None] * self.config.num_hidden_layers
        )[0]
        return self._classify(out[:, 0, :], labels)


# ─── Condition B: same backbone, standard text attention (no DFG) ─────────────
class ModelNoDFG(ClassificationModel):
    def forward(self, input_ids=None, attention_mask=None, labels=None):
        # Standard forward — GraphCodeBERT backbone, but just use it like RoBERTa
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)[0]
        return self._classify(out[:, 0, :], labels)

In [5]:
# ─── DATASET A — with DFG (full GraphCodeBERT format) ────────────────────────
class TextDataset(Dataset):
    def __init__(self, tokenizer, args, file_path):
        self.args      = args
        self.tokenizer = tokenizer
        self.total_len = args.code_length + args.data_flow_length
        with open(file_path) as f:
            self.lines = f.readlines()

    def __len__(self): return len(self.lines)

    def _char_idx(self, lines, coord):
        r, c = coord
        return sum(len(lines[i]) for i in range(min(r, len(lines)))) + c

    def __getitem__(self, idx):
        entry      = json.loads(self.lines[idx])
        code       = entry.get('code', '')
        dfg        = entry.get('dfg', [])[:self.args.data_flow_length]
        label      = int(entry.get('label', 0))
        code_lines = code.splitlines(keepends=True)

        tok = self.tokenizer(
            code, max_length=self.args.code_length, truncation=True,
            padding='max_length', return_offsets_mapping=True
        )
        input_ids, offsets = tok['input_ids'], tok['offset_mapping']
        dfg_ids = [self.tokenizer.unk_token_id] * len(dfg)
        n2t, p2n = {}, {}

        for ni, node in enumerate(dfg):
            sp, ep = node[1][0], node[1][1]
            pk = (sp[0], sp[1], ep[0], ep[1])
            p2n[pk] = ni
            try:
                cs = self._char_idx(code_lines, sp)
                ce = self._char_idx(code_lines, ep)
                n2t[ni] = [i for i, (ts, te) in enumerate(offsets)
                           if ts != te and
                           ((ts >= cs and te <= ce) or (cs >= ts and ce <= te))]
            except IndexError:
                n2t[ni] = []

        c_len = self.args.code_length
        mask  = np.zeros((self.total_len, self.total_len), dtype=bool)
        mask[:c_len, :c_len] = True
        for ni, node in enumerate(dfg):
            ani = c_len + ni
            for ti in n2t.get(ni, []):
                mask[ani, ti] = mask[ti, ani] = True
            for pp in node[4]:
                pk = (pp[0][0], pp[0][1], pp[1][0], pp[1][1])
                if pk in p2n:
                    mask[ani, c_len + p2n[pk]] = mask[c_len + p2n[pk], ani] = True
            mask[ani, ani] = True

        ids   = input_ids + dfg_ids
        p_ids = [i + 2 for i in range(c_len)] + [0] * len(dfg_ids)
        pad   = self.total_len - len(ids)
        if pad > 0:
            ids   += [self.tokenizer.pad_token_id] * pad
            p_ids += [1] * pad

        return {
            'input_ids': torch.tensor(ids,          dtype=torch.long),
            'p_ids':     torch.tensor(p_ids,        dtype=torch.long),
            'attn_mask': torch.tensor(mask,         dtype=torch.float),
            'label':     torch.tensor(label,        dtype=torch.long)
        }


# ─── DATASET B — code only (no DFG tokens appended) ──────────────────────────
class TextDatasetNoDFG(Dataset):
    """Same underlying file, but strips DFG — uses GraphCodeBERT tokenizer."""
    def __init__(self, tokenizer, args, file_path):
        self.tokenizer = tokenizer
        self.args      = args
        with open(file_path) as f:
            self.lines = f.readlines()

    def __len__(self): return len(self.lines)

    def __getitem__(self, idx):
        try:
            entry = json.loads(self.lines[idx])
        except Exception:
            return self.__getitem__((idx + 1) % len(self.lines))
        code  = entry.get('code', '')
        label = int(entry.get('label', 0))
        tok   = self.tokenizer(
            code,
            max_length=self.args.code_length,  # code_length only — no DFG slots
            truncation=True,
            padding='max_length'
        )
        return {
            'input_ids':      torch.tensor(tok['input_ids'],      dtype=torch.long),
            'attention_mask': torch.tensor(tok['attention_mask'], dtype=torch.long),
            'label':          torch.tensor(label,                 dtype=torch.long)
        }

In [6]:
# ─── EVALUATION HELPERS ───────────────────────────────────────────────────────
@torch.no_grad()
def eval_with_dfg(model, dataset, desc="Eval (DFG)"):
    loader = DataLoader(dataset, batch_size=args.eval_batch_size,
                        num_workers=2, pin_memory=True)
    model.eval()
    preds, labels, probs_list = [], [], []
    for batch in tqdm(loader, desc=desc):
        inp = {
            'input_ids': batch['input_ids'].to(args.device),
            'p_ids':     batch['p_ids'].to(args.device),
            'attn_mask': batch['attn_mask'].to(args.device)
        }
        prob = model(**inp)
        probs_list.extend(prob[:, 1].cpu().numpy())
        preds.extend(torch.argmax(prob, dim=-1).cpu().numpy())
        labels.extend(batch['label'].numpy())
    model.train()
    return preds, labels, np.array(probs_list)


@torch.no_grad()
def eval_no_dfg(model, dataset, desc="Eval (no DFG)"):
    loader = DataLoader(dataset, batch_size=args.eval_batch_size,
                        num_workers=2, pin_memory=True)
    model.eval()
    preds, labels, probs_list = [], [], []
    for batch in tqdm(loader, desc=desc):
        inp = {
            'input_ids':      batch['input_ids'].to(args.device),
            'attention_mask': batch['attention_mask'].to(args.device)
        }
        prob = model(**inp)
        probs_list.extend(prob[:, 1].cpu().numpy())
        preds.extend(torch.argmax(prob, dim=-1).cpu().numpy())
        labels.extend(batch['label'].numpy())
    model.train()
    return preds, labels, np.array(probs_list)


def report(name, preds, labels, probs):
    acc     = accuracy_score(labels, preds)
    roc_auc = roc_auc_score(labels, probs)
    pr_auc  = average_precision_score(labels, probs)
    cm      = confusion_matrix(labels, preds)
    tn, fp, fn, tp = cm.ravel()

    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(f"  Accuracy : {acc:.4%}")
    print(f"  ROC-AUC  : {roc_auc:.4f}")
    print(f"  PR-AUC   : {pr_auc:.4f}")
    print(f"  FN (missed malware): {fn}")
    print()
    print(classification_report(
        labels, preds, target_names=['Safe', 'Malicious'], digits=4
    ))
    return {'name': name, 'acc': acc, 'roc_auc': roc_auc,
            'pr_auc': pr_auc, 'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp}

In [7]:
# ─── TRAIN NO-DFG MODEL (Condition B) ────────────────────────────────────────
# We train a fresh head on top of the same GraphCodeBERT backbone
# but WITHOUT DFG tokens or the custom attention mask.

def train_no_dfg(model, train_ds, val_ds_no_dfg, tokenizer):
    loader    = DataLoader(train_ds, batch_size=args.train_batch_size,
                           shuffle=True, num_workers=2, pin_memory=True)
    optimizer = AdamW(model.parameters(), lr=args.learning_rate, eps=1e-8)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=0,
        num_training_steps=len(loader) * args.num_train_epochs
    )
    scaler    = GradScaler()
    best_acc  = 0.0

    for epoch in range(args.num_train_epochs):
        model.train()
        tr_loss = 0.0
        for batch in tqdm(loader, desc=f"NoDFG Epoch {epoch}"):
            inp = {
                'input_ids':      batch['input_ids'].to(args.device),
                'attention_mask': batch['attention_mask'].to(args.device),
                'labels':         batch['label'].to(args.device)
            }
            optimizer.zero_grad()
            with autocast():
                loss, _ = model(**inp)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            tr_loss += loss.item()

        val_preds, val_labels, _ = eval_no_dfg(model, val_ds_no_dfg,
                                               desc=f"  NoDFG Val {epoch}")
        val_acc = accuracy_score(val_labels, val_preds)
        print(f"Epoch {epoch} | Loss: {tr_loss/len(loader):.4f} | Val Acc: {val_acc:.4%}")

        if val_acc > best_acc:
            best_acc = val_acc
            os.makedirs(args.output_dir_nodeg, exist_ok=True)
            torch.save(model.state_dict(),
                       os.path.join(args.output_dir_nodeg, "model.bin"))
            print(f"  ✓ New best: {best_acc:.4%}")

    print(f"No-DFG training done. Best val acc: {best_acc:.4%}")

In [8]:
# ─── MAIN ─────────────────────────────────────────────────────────────────────
set_seed(args.seed)

# ── 1. Shared tokenizer (both conditions use GraphCodeBERT tokenizer) ──
config    = RobertaConfig.from_pretrained(args.model_name_or_path)
config.num_labels = 2
tokenizer = AutoTokenizer.from_pretrained(args.tokenizer_name)

# ── 2. Condition A — Load trained GraphCodeBERT+DFG ──────────────────
print("[Condition A] Loading GraphCodeBERT + DFG model...")
enc_a   = RobertaModel.from_pretrained(args.model_name_or_path, config=config)
model_a = ModelWithDFG(enc_a, config).to(args.device)
model_a.load_state_dict(torch.load(args.gcb_weights, map_location=args.device))
model_a.eval()
print("  ✓ Loaded")

# ── 3. Build datasets ─────────────────────────────────────────────────
print("\nBuilding datasets...")
ds_dfg    = TextDataset(tokenizer, args, args.train_file)
ds_no_dfg = TextDatasetNoDFG(tokenizer, args, args.train_file)
n         = len(ds_dfg)
train_n, val_n = int(0.9 * n), n - int(0.9 * n)

# Use the SAME seed so both splits select identical indices
gen_a = torch.Generator().manual_seed(args.seed)
_, val_ds_dfg = random_split(ds_dfg, [train_n, val_n], generator=gen_a)

gen_b = torch.Generator().manual_seed(args.seed)
train_ds_no_dfg, val_ds_no_dfg = random_split(
    ds_no_dfg, [train_n, val_n], generator=gen_b
)
print(f"Val samples: {val_n:,}")

# ── 4. Condition B — Train no-DFG model ──────────────────────────────
no_dfg_ckpt = os.path.join(args.output_dir_nodeg, "model.bin")
print("\n[Condition B] Training GraphCodeBERT (no DFG)...")
enc_b   = RobertaModel.from_pretrained(args.model_name_or_path, config=config)
model_b = ModelNoDFG(enc_b, config).to(args.device)

if os.path.exists(no_dfg_ckpt):
    print(f"  Found existing checkpoint at {no_dfg_ckpt}. Loading...")
    model_b.load_state_dict(torch.load(no_dfg_ckpt, map_location=args.device))
else:
    train_no_dfg(model_b, train_ds_no_dfg, val_ds_no_dfg, tokenizer)
    model_b.load_state_dict(torch.load(no_dfg_ckpt, map_location=args.device))

model_b.eval()
print("  ✓ No-DFG model ready")

config.json:   0%|          | 0.00/539 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

...

Epoch 0 | Loss: 0.3075 | Val Acc: 87.3475%
  ✓ New best: 87.3475%


NoDFG Epoch 1:   0%|          | 0/11248 [00:00<?, ?it/s]/tmp/ipykernel_24/2597288154.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
  NoDFG Val 1: 100%|██████████| 625/625 [03:48<00:00,  2.74it/s]


Epoch 1 | Loss: 0.2494 | Val Acc: 88.1076%
  ✓ New best: 88.1076%


NoDFG Epoch 2:   0%|          | 0/11248 [00:00<?, ?it/s]/tmp/ipykernel_24/2597288154.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
  NoDFG Val 2: 100%|██████████| 625/625 [03:48<00:00,  2.74it/s]


Epoch 2 | Loss: 0.2105 | Val Acc: 88.7127%
  ✓ New best: 88.7127%
No-DFG training done. Best val acc: 88.7127%
  ✓ No-DFG model ready


In [9]:
# ─── EVALUATE BOTH CONDITIONS ─────────────────────────────────────────────────
print("\nEvaluating Condition A (GraphCodeBERT + DFG)...")
preds_a, labels_a, probs_a = eval_with_dfg(model_a, val_ds_dfg)

print("Evaluating Condition B (GraphCodeBERT, no DFG)...")
preds_b, labels_b, probs_b = eval_no_dfg(model_b, val_ds_no_dfg)

res_a = report("Condition A: GraphCodeBERT + DFG", preds_a, labels_a, probs_a)
res_b = report("Condition B: GraphCodeBERT (no DFG)", preds_b, labels_b, probs_b)


Evaluating Condition A (GraphCodeBERT + DFG)...


Eval (DFG): 100%|██████████| 625/625 [05:13<00:00,  2.00it/s]


Evaluating Condition B (GraphCodeBERT, no DFG)...


Eval (no DFG): 100%|██████████| 625/625 [03:48<00:00,  2.74it/s]



  Condition A: GraphCodeBERT + DFG
  Accuracy : 91.8184%
  ROC-AUC  : 0.9798
  PR-AUC   : 0.9797
  FN (missed malware): 829

              precision    recall  f1-score   support

        Safe     0.9181    0.9201    0.9191     10095
   Malicious     0.9183    0.9163    0.9173      9901

    accuracy                         0.9182     19996
   macro avg     0.9182    0.9182    0.9182     19996
weighted avg     0.9182    0.9182    0.9182     19996


  Condition B: GraphCodeBERT (no DFG)
  Accuracy : 88.7127%
  ROC-AUC  : 0.9615
  PR-AUC   : 0.9619
  FN (missed malware): 1111

              precision    recall  f1-score   support

        Safe     0.8896    0.8865    0.8880     10095
   Malicious     0.8847    0.8878    0.8862      9901

    accuracy                         0.8871     19996
   macro avg     0.8871    0.8871    0.8871     19996
weighted avg     0.8871    0.8871    0.8871     19996



In [10]:
# ─── ABLATION DELTA SUMMARY ───────────────────────────────────────────────────
print("\n" + "="*55)
print("ABLATION DELTA:  A (DFG)  vs  B (no DFG)")
print("="*55)

metrics = ['acc', 'roc_auc', 'pr_auc']
labels_str = ['Accuracy', 'ROC-AUC', 'PR-AUC']
for m, lbl in zip(metrics, labels_str):
    delta = res_a[m] - res_b[m]
    arrow = '▲' if delta > 0 else '▼'
    print(f"  {lbl:10s}: A={res_a[m]:.4f}  B={res_b[m]:.4f}  "
          f"Δ={delta:+.4f} {arrow}")

fn_delta = res_a['fn'] - res_b['fn']
print(f"\n  False Negatives (missed malware):")
print(f"    A (DFG):    {res_a['fn']}")
print(f"    B (no DFG): {res_b['fn']}")
print(f"    Δ (A - B):  {fn_delta:+d}")

if res_a['acc'] > res_b['acc']:
    print("\n✓ DFG attention HELPS: Condition A outperforms Condition B.")
else:
    print("\n✗ DFG attention does NOT help, or hurts performance.")


ABLATION DELTA:  A (DFG)  vs  B (no DFG)
  Accuracy  : A=0.9182  B=0.8871  Δ=+0.0311 ▲
  ROC-AUC   : A=0.9798  B=0.9615  Δ=+0.0183 ▲
  PR-AUC    : A=0.9797  B=0.9619  Δ=+0.0178 ▲

  False Negatives (missed malware):
    A (DFG):    829
    B (no DFG): 1111
    Δ (A - B):  -282

✓ DFG attention HELPS: Condition A outperforms Condition B.


In [11]:
# ─── BAR CHART COMPARISON ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 5))
conditions = ['GraphCodeBERT\n+ DFG', 'GraphCodeBERT\n(no DFG)']
colors     = ['#4C72B0', '#DD8452']

metric_info = [
    ('Accuracy', [res_a['acc'],     res_b['acc']]),
    ('ROC-AUC',  [res_a['roc_auc'], res_b['roc_auc']]),
    ('PR-AUC',   [res_a['pr_auc'],  res_b['pr_auc']]),
]

for ax, (title, vals) in zip(axes, metric_info):
    bars = ax.bar(conditions, vals, color=colors, width=0.5, edgecolor='white')
    ymin = max(0, min(vals) - 0.05)
    ax.set_ylim([ymin, min(1.0, max(vals) + 0.04)])
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylabel(title, fontsize=11)
    ax.tick_params(axis='x', labelsize=9)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height() + 0.002,
                f'{val:.4f}', ha='center', va='bottom', fontsize=10)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Ablation Study: DFG vs No-DFG on GraphCodeBERT',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plot_path = "/kaggle/working/test3_ablation_bar.png"
fig.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"Bar chart saved → {plot_path}")

Bar chart saved → /kaggle/working/test3_ablation_bar.png


In [12]:
# ─── SAVE RESULTS ────────────────────────────────────────────────────────────
out_path = "/kaggle/working/test3_ablation_results.txt"
with open(out_path, "w") as f:
    f.write("Test 3: DFG Ablation Study Results\n")
    f.write("=" * 55 + "\n\n")
    for res in [res_a, res_b]:
        f.write(f"{res['name']}\n")
        f.write(f"  Accuracy : {res['acc']:.4%}\n")
        f.write(f"  ROC-AUC  : {res['roc_auc']:.4f}\n")
        f.write(f"  PR-AUC   : {res['pr_auc']:.4f}\n")
        f.write(f"  TP={res['tp']}  FP={res['fp']}  "
                f"FN={res['fn']}  TN={res['tn']}\n\n")
    f.write("Delta (A - B):\n")
    for m, lbl in zip(metrics, labels_str):
        f.write(f"  {lbl}: {res_a[m] - res_b[m]:+.4f}\n")

print(f"Results saved → {out_path}")
print(f"Bar chart   → /kaggle/working/test3_ablation_bar.png")

Results saved → /kaggle/working/test3_ablation_results.txt
Bar chart   → /kaggle/working/test3_ablation_bar.png
